# Morningstar-European-ETFs - Preprocessing

In [ ]:
import pandas as pd
import xgboost as xgb
import shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import warnings

pd.set_option('display.max_rows', None)
warnings.simplefilter(action='ignore', category=FutureWarning)


In [ ]:
etf_etf_df_clean = pd.read_csv("Morningstar-European-ETFs.csv")

## 1. Exploratory Data Analysis EDA

In [ ]:
etf_etf_df_clean.head()

In [ ]:
etf_etf_df_clean.describe()

In [ ]:
etf_etf_df_clean.info()

In [ ]:
etf_etf_df_clean.isnull().sum()

## 2. Data Preprocessing

In [ ]:
# Selecionando colunas categ?ricas
categorical_cols = etf_etf_df_clean.select_dtypes(include=['object']).columns.tolist()

# Identificando colunas categ?ricas com qualquer valor faltante
cols_with_missing_values = [col for col in categorical_cols if etf_etf_df_clean[col].isnull().sum() > 0]

print(f"Antes: {etf_etf_df_clean.shape[1]} colunas")

# Removendo colunas categ?ricas com valores faltantes
etf_df_clean = etf_etf_df_clean.drop(columns=cols_with_missing_values)
print(f"Depois de remover colunas categ?ricas com NaNs: {etf_df_clean.shape[1]} colunas")

# Definindo o limite de dados faltantes para 50% do tamanho do dataset
threshold = 0.5 * len(etf_df_clean)

# Removendo colunas com mais de 50% de dados faltantes
etf_df_clean = etf_df_clean.dropna(thresh=threshold, axis=1)
print(f"Depois de remover colunas com mais de 50% NaNs: {etf_df_clean.shape[1]} colunas")
print(etf_df_clean.isnull().sum())


In [ ]:
# Para colunas setoriais, imputar NaNs com 0, pois o ETF n?o possui exposi??o setorial.
sector_columns = [col for col in etf_df_clean.columns if 'sector_' in col]
etf_df_clean[sector_columns] = etf_df_clean[sector_columns].fillna(0)

# Imputando colunas de rating e scores com a moda
rating_cols = ['rating', 'risk_rating', 'performance_rating', 'equity_style_score', 'equity_size_score']
for col in rating_cols:
    if col in etf_df_clean.columns:
        mode_val = etf_df_clean[col].mode()[0]
        etf_df_clean[col].fillna(mode_val, inplace=True)

# Imputando colunas num?ricas com a mediana
numeric_cols = [
    'price_prospective_earnings', 'price_book_ratio', 'price_sales_ratio',
    'price_cash_flow_ratio', 'dividend_yield_factor',
    'long_term_projected_earnings_growth', 'historical_earnings_growth',
    'sales_growth', 'cash_flow_growth', 'book_value_growth',
    'roa', 'roe', 'roic'
]
for col in numeric_cols:
    if col in etf_df_clean.columns:
        median_val = etf_df_clean[col].median()
        etf_df_clean[col].fillna(median_val, inplace=True)

# Imputando colunas de envolvimento com 0
involvement_cols = [
    'involvement_abortive_contraceptive', 'involvement_alcohol',
    'involvement_animal_testing', 'involvement_controversial_weapons',
    'involvement_gambling', 'involvement_gmo', 
    'involvement_military_contracting', 'involvement_nuclear',
    'involvement_palm_oil', 'involvement_pesticides',
    'involvement_small_arms', 'involvement_thermal_coal', 'involvement_tobacco'
]
etf_df_clean[involvement_cols] = etf_df_clean[involvement_cols].fillna(0)

# Imputando colunas de tamanho e valor com a mediana
size_value_cols = ['nav_per_share', 'shareclass_size', 'fund_size']
for col in size_value_cols:
    if col in etf_df_clean.columns:
        median_val = etf_df_clean[col].median()
        etf_df_clean[col].fillna(median_val, inplace=True)

# Imputar com zero para colunas de capitaliza??o de mercado
market_cap_cols = ['market_cap_giant', 'market_cap_large', 'market_cap_medium', 'market_cap_small', 'market_cap_micro']
etf_df_clean[market_cap_cols] = etf_df_clean[market_cap_cols].fillna(0)

# Imputar com mediana para custos associados ao ETF
cost_cols = ['ongoing_cost', 'management_fees']
for col in cost_cols:
    if col in etf_df_clean.columns:
        median_val = etf_df_clean[col].median()
        etf_df_clean[col].fillna(median_val, inplace=True)

# Imputar com zero para m?tricas de sustentabilidade
sustainability_cols = ['environmental_score', 'social_score', 'governance_score', 'sustainability_score']
etf_df_clean[sustainability_cols] = etf_df_clean[sustainability_cols].fillna(0)

# Excluir colunas de ranking de sustentabilidade
etf_df_clean.drop(['sustainability_rank', 'sustainability_percentage_rank'], axis=1, inplace=True, errors='ignore')


In [ ]:
print(etf_df_clean.isnull().sum())

In [ ]:
# Lista das colunas de retorno dos fundos
fund_return_cols = [
    'fund_trailing_return_ytd',
    'fund_trailing_return_3years',
    'fund_trailing_return_5years',
    'fund_return_2019',
    'fund_return_2018',
    'fund_return_2017',
    'fund_return_2016',
    'fund_return_2020_q3',
    'fund_return_2020_q2',
    'fund_return_2020_q1',
    'fund_return_2019_q4',
    'fund_return_2019_q3',
    'fund_return_2019_q2',
    'fund_return_2019_q1',
    'fund_return_2018_q4',
    'fund_return_2018_q3',
    'fund_return_2018_q2',
    'fund_return_2018_q1',
    'fund_return_2017_q4',
    'fund_return_2017_q3',
    'fund_return_2017_q2',
    'fund_return_2017_q1',
    'fund_return_2016_q4',
    'fund_return_2016_q3',
    'fund_return_2016_q2',
    'fund_return_2016_q1',
    'fund_return_2015_q4'
]

# Excluindo linhas que cont?m NaN em qualquer uma das colunas de retorno dos fundos
etf_df_clean = etf_df_clean.dropna(subset=fund_return_cols)

# Imprimindo o n?mero de linhas ap?s a exclus?o
print(f"Total de linhas ap?s exclus?o: {etf_df_clean.shape[0]}")


In [ ]:
etf_df_clean.head()

In [ ]:
print(etf_df_clean.isnull().sum())

In [ ]:
# Salvando o DataFrame processado em um arquivo CSV
etf_df_clean.to_csv('Morningstar-European-ETFs-Processed.csv', index=False)

## 3. Feature importance Analysis

In [ ]:
# Calcular a correla??o de Pearson em rela??o ao target (risk_rating)
etf_df_clean = pd.read_csv("Morningstar-European-ETFs-Processed.csv")
correlations = etf_df_clean.corr(method='pearson')['risk_rating'].sort_values(ascending=False)
print(correlations)

In [ ]:
# Codifica??o One-Hot
etf_df_clean = pd.get_dummies(etf_df_clean)

# Dividir os dados em treinamento e teste
X = etf_df_clean.drop('risk_rating', axis=1)
y = etf_df_clean['risk_rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar o modelo XGBoost
model_xgb = xgb.XGBRegressor()
model_xgb.fit(X_train, y_train)

# Usar SHAP para decompor a import?ncia das caracter?sticas
explainer = shap.Explainer(model_xgb)
shap_values = explainer.shap_values(X_train)

# Visualizar a import?ncia das caracter?sticas com SHAP
shap.summary_plot(shap_values, X_train)


In [ ]:
# Lista das colunas
cols = [
    'risk_rating',
    'fund_return_2020_q1',
    'rating',
    'sector_consumer_defensive',
    'fund_trailing_return_ytd',
    'performance_rating',
    'fund_return_2018_q3',
    'fund_return_2018',
    'fund_return_2016_q4',
    'involvement_animal_testing',
    'price_prospective_earnings'
]

# Dicion?rio de descri??es (para melhorar os prints)
descriptions = {
    'risk_rating': 'Classifica??o de risco do ETF.',
    'fund_return_2020_q1': 'Retorno do fundo no primeiro trimestre de 2020.',
    'rating': 'Classifica??o geral do ETF.',
    'sector_consumer_defensive': 'Exposi??o do ETF ao setor de consumo defensivo.',
    'fund_trailing_return_ytd': 'Retorno acumulado do fundo no ano at? a data.',
    'performance_rating': 'Classifica??o de performance do ETF.',
    'fund_return_2018_q3': 'Retorno do fundo no terceiro trimestre de 2018.',
    'fund_return_2018': 'Retorno total do fundo em 2018.',
    'fund_return_2016_q4': 'Retorno do fundo no quarto trimestre de 2016.',
    'involvement_animal_testing': 'Envolvimento do ETF em testes em animais.',
    'price_prospective_earnings': 'Rela??o pre?o/perspectiva de ganhos do ETF.'
}

for col in cols:
    print(f" Coluna: {col}")
    print(f"   - Representa: {descriptions[col]}")
    print(f"   - Tipo de dados: {etf_df_clean[col].dtype}")
    print(f"   - Valores v?lidos: (Baseie-se no conhecimento do dom?nio ou na documenta??o dispon?vel sobre os dados.)")
    print(f"   - Valores distintos: {etf_df_clean[col].nunique()}")
    print(f"   - Menor e maior valor: {etf_df_clean[col].min()} e {etf_df_clean[col].max()}")
    print(f"   - Moda: {etf_df_clean[col].mode().iloc[0]}")  
    print(f"   - M?dia: {etf_df_clean[col].mean():.2f}")  
    print(f"   - Desvio-padr?o: {etf_df_clean[col].std():.2f}")  
    print(f"   - Mediana: {etf_df_clean[col].median()}")
    print("\n")  # Espa?amento entre colunas
